In [387]:






import pandas as pd
from pathlib import Path

import os

PROJECT_ROOT = Path(
    os.getenv("PROJECT_ROOT", Path.cwd())
).expanduser().resolve()


JUDGE_PATH16 = (
    PROJECT_ROOT


    / "llm_judge_results_final_combined.csv"

)


CSV_PATH_NEW16 =  Path(os.getenv("ANSWERS_LOG_PATH", str(JUDGE_PATH16))).expanduser().resolve()
df_newest_combi = pd.read_csv(CSV_PATH_NEW16)

In [395]:
SCRIPT_COL ="script"

In [388]:
import pandas as pd

# -------------------------------
# 0) Helpers / config
# -------------------------------
METRIC_COLS = {
    "Answer_Relevance": "answer_relevance_1to5",
    "Completeness": "completeness_1to5",
    
    "Helpfulness": "helpfulness_final_1to5",
}

CAT_COL = "correctness_category"

QTYPE_COL = "query_type"


metrics_per_script = (
    df_newest_combi
    .groupby(SCRIPT_COL)
    .agg(
        Answer_Relevance=(METRIC_COLS["Answer_Relevance"], "mean"),
        Completeness=(METRIC_COLS["Completeness"], "mean"),
       
        Helpfulness=(METRIC_COLS["Helpfulness"], "mean"),
    )
    .round(2)
)


cat_counts_per_script = (
    df_newest_combi
    .groupby(SCRIPT_COL)[CAT_COL]
    .value_counts(dropna=False)
    .unstack(fill_value=0)
)

# Optional: Spalten sicher in gewünschter Reihenfolge
for c in ["TP", "FP", "FN", "PARTIAL"]:
    if c not in cat_counts_per_script.columns:
        cat_counts_per_script[c] = 0
cat_counts_per_script = cat_counts_per_script[["TP", "FP", "FN", "PARTIAL"]]

summary_per_script = metrics_per_script.join(cat_counts_per_script)


summary_per_script




,Answer_Relevance,Completeness,Helpfulness,TP,FP,FN,PARTIAL
script,,,,,,,
LLMGraph_Community_Hybrid,3.70,3.02,4.22,108,10,14,28
LLMGraph_Dense_KG,3.71,2.74,4.51,99,7,22,32
LLMGraph_Hybrid_KG,3.79,2.99,4.31,118,12,11,19
LLMGraph_Hybrid_KG_Reranker,3.72,2.96,4.38,110,12,16,22
LlamaIndex_Community_Only,3.60,2.52,3.85,74,16,35,35
LlamaIndex_Dense_KG,3.70,2.46,4.17,83,11,42,24
LlamaIndex_Hybrid_KG_Rerank,3.71,3.06,4.59,107,12,14,27
RAG_Advanced_Dense,3.42,1.42,2.00,9,21,110,20
RAG_Advanced_Hybrid,3.75,2.88,4.56,99,15,10,36


In [389]:
import numpy as np


df = summary_per_script.copy()

TP = df["TP"]
FP = df["FP"]
FN = df["FN"]
PARTIAL = df["PARTIAL"]

# -------------------------------
# HARD Precision & Recall
# (PARTIAL zählt als falsch)
# -------------------------------
df["Precision"] = np.where(
    (TP + FP + PARTIAL) > 0,
    TP / (TP + FP + PARTIAL),
    0.0
)

df["Recall"] = np.where(
    (TP + FN + PARTIAL) > 0,
    TP / (TP + FN + PARTIAL),
    0.0
)

df[["Precision", "Recall"]] = (
    df[["Precision", "Recall"]]
    .round(3)
)

result = df[
    [
        "Answer_Relevance",
        "Completeness",
        "Helpfulness",
        "Precision",
        "Recall",
    ]
]

result.sort_values(by="script", ascending=True)



,Answer_Relevance,Completeness,Helpfulness,Precision,Recall
script,,,,,
LLMGraph_Community_Hybrid,3.70,3.02,4.22,0.740,0.720
LLMGraph_Dense_KG,3.71,2.74,4.51,0.717,0.647
LLMGraph_Hybrid_KG,3.79,2.99,4.31,0.792,0.797
LLMGraph_Hybrid_KG_Reranker,3.72,2.96,4.38,0.764,0.743
LlamaIndex_Community_Only,3.60,2.52,3.85,0.592,0.514
LlamaIndex_Dense_KG,3.70,2.46,4.17,0.703,0.557
LlamaIndex_Hybrid_KG_Rerank,3.71,3.06,4.59,0.733,0.723
RAG_Advanced_Dense,3.42,1.42,2.00,0.180,0.065
RAG_Advanced_Hybrid,3.75,2.88,4.56,0.660,0.683


In [390]:
import numpy as np

# 1) Counts pro script & query_type
counts = (
    df_newest_combi
    .groupby([SCRIPT_COL, QTYPE_COL])[CAT_COL]
    .value_counts(dropna=False)
    .unstack(fill_value=0)
)

# Ensure required columns exist
for c in ["TP", "FN", "PARTIAL"]:
    if c not in counts.columns:
        counts[c] = 0

counts = counts[["TP", "FN", "PARTIAL"]]

# 2) Hard Recall (PARTIAL zählt als falsch)
counts["Recall"] = np.where(
    (counts["TP"] + counts["FN"] + counts["PARTIAL"]) > 0,
    counts["TP"] / (counts["TP"] + counts["FN"] + counts["PARTIAL"]),
    0.0
)

counts["Recall"] = counts["Recall"].round(3)

# 3) Pivot: script x query_type
hard_recall_matrix = (
    counts["Recall"]
    .unstack(QTYPE_COL)
    .fillna(0)
)

hard_recall_matrix.sort_values(by="script", ascending=True)


query_type,disambiguation,factual,multi_hop,reasoning,summary
script,,,,,
LLMGraph_Community_Hybrid,0.731,0.690,0.724,0.724,0.730
LLMGraph_Dense_KG,0.741,0.633,0.621,0.733,0.541
LLMGraph_Hybrid_KG,0.875,0.714,0.828,0.857,0.744
LLMGraph_Hybrid_KG_Reranker,0.808,0.643,0.793,0.793,0.694
LlamaIndex_Community_Only,0.455,0.393,0.467,0.680,0.564
LlamaIndex_Dense_KG,0.654,0.433,0.679,0.571,0.486
LlamaIndex_Hybrid_KG_Rerank,0.880,0.679,0.767,0.750,0.595
RAG_Advanced_Dense,0.045,0.033,0.069,0.077,0.094
RAG_Advanced_Hybrid,0.846,0.900,0.571,0.769,0.400


In [391]:
# -------------------------------

# -------------------------------
def pivot_metric_by_script_and_query_type(
    df: pd.DataFrame,
    metric_name: str,
    aggfunc="mean",
    round_digits: int = 2,
) -> pd.DataFrame:
    """
    metric_name: one of
      - "Answer_Relevance"
      - "Completeness"
      - "Correctness"
      - "Helpfulness"
    """
    if metric_name not in METRIC_COLS:
        raise ValueError(f"Unknown metric_name='{metric_name}'. Use: {list(METRIC_COLS.keys())}")

    col = METRIC_COLS[metric_name]

    pivot = (
        df.pivot_table(
            index=SCRIPT_COL,
            columns=QTYPE_COL,
            values=col,
            aggfunc=aggfunc,
        )
        .round(round_digits)
        .sort_index()
    )
    return pivot


# Beispiel: eine Metrik auswählen
metric_name = "Answer_Relevance"  # <- hier jeweils ändern, z.B. "Correctness"
pivot_df = pivot_metric_by_script_and_query_type(df_newest_combi, metric_name)

print(f"\n=== Pivot: {metric_name} (rows=script, cols=query_type) ===")
pivot_df


# -------------------------------
# 3) Optional: Pivot der correctness_category Counts (TP/FP/FN/PARTIAL) je script x query_type
# -------------------------------




=== Pivot: Answer_Relevance (rows=script, cols=query_type) ===


query_type,disambiguation,factual,multi_hop,reasoning,summary
script,,,,,
LLMGraph_Community_Hybrid,3.60,3.87,3.53,3.73,3.75
LLMGraph_Dense_KG,3.63,3.87,3.67,3.57,3.80
LLMGraph_Hybrid_KG,3.73,4.13,3.67,3.57,3.85
LLMGraph_Hybrid_KG_Reranker,3.80,4.10,3.60,3.53,3.62
LlamaIndex_Community_Only,3.63,3.53,3.57,3.37,3.82
LlamaIndex_Dense_KG,3.83,3.83,3.73,3.43,3.68
LlamaIndex_Hybrid_KG_Rerank,3.63,3.97,3.73,3.50,3.72
RAG_Advanced_Dense,3.33,3.37,3.43,3.43,3.50
RAG_Advanced_Hybrid,3.80,4.00,3.57,3.77,3.65


In [392]:
# -------------------------------

# -------------------------------
def pivot_metric_by_script_and_query_type(
    df: pd.DataFrame,
    metric_name: str,
    aggfunc="mean",
    round_digits: int = 2,
) -> pd.DataFrame:
    """
    metric_name: one of
      - "Answer_Relevance"
      - "Completeness"
      - "Correctness"
      - "Helpfulness"
    """
    if metric_name not in METRIC_COLS:
        raise ValueError(f"Unknown metric_name='{metric_name}'. Use: {list(METRIC_COLS.keys())}")

    col = METRIC_COLS[metric_name]

    pivot = (
        df.pivot_table(
            index=SCRIPT_COL,
            columns=QTYPE_COL,
            values=col,
            aggfunc=aggfunc,
        )
        .round(round_digits)
        .sort_index()
    )
    return pivot


# Beispiel: eine Metrik auswählen
metric_name = "Completeness"  # <- hier jeweils ändern, z.B. "Correctness"
pivot_df = pivot_metric_by_script_and_query_type(df_newest_combi, metric_name)

print(f"\n=== Pivot: {metric_name} (rows=script, cols=query_type) ===")
pivot_df


# -------------------------------
# 3) Optional: Pivot der correctness_category Counts (TP/FP/FN/PARTIAL) je script x query_type
# -------------------------------




=== Pivot: Completeness (rows=script, cols=query_type) ===


query_type,disambiguation,factual,multi_hop,reasoning,summary
script,,,,,
LLMGraph_Community_Hybrid,3.43,3.57,2.77,2.90,2.60
LLMGraph_Dense_KG,2.53,3.13,2.90,2.60,2.60
LLMGraph_Hybrid_KG,2.80,3.50,2.77,3.00,2.92
LLMGraph_Hybrid_KG_Reranker,2.93,3.47,2.87,2.97,2.65
LlamaIndex_Community_Only,2.67,2.53,2.43,2.47,2.50
LlamaIndex_Dense_KG,2.60,2.60,2.43,2.23,2.45
LlamaIndex_Hybrid_KG_Rerank,3.03,3.37,3.30,2.90,2.78
RAG_Advanced_Dense,1.30,1.13,1.60,1.37,1.65
RAG_Advanced_Hybrid,3.23,3.23,3.07,2.50,2.50


In [393]:

def pivot_metric_by_script_and_query_type(
    df: pd.DataFrame,
    metric_name: str,
    aggfunc="mean",
    round_digits: int = 2,
) -> pd.DataFrame:
    """
    metric_name: one of
      - "Answer_Relevance"
      - "Completeness"
      - "Correctness"
      - "Helpfulness"
    """
    if metric_name not in METRIC_COLS:
        raise ValueError(f"Unknown metric_name='{metric_name}'. Use: {list(METRIC_COLS.keys())}")

    col = METRIC_COLS[metric_name]

    pivot = (
        df.pivot_table(
            index=SCRIPT_COL,
            columns=QTYPE_COL,
            values=col,
            aggfunc=aggfunc,
        )
        .round(round_digits)
        .sort_index()
    )
    return pivot



metric_name = "Helpfulness"  
pivot_df = pivot_metric_by_script_and_query_type(df_newest_combi, metric_name)

print(f"\n=== Pivot: {metric_name} (rows=script, cols=query_type) ===")
pivot_df





=== Pivot: Helpfulness (rows=script, cols=query_type) ===


query_type,disambiguation,factual,multi_hop,reasoning,summary
script,,,,,
LLMGraph_Community_Hybrid,4.23,4.20,4.33,4.20,4.15
LLMGraph_Dense_KG,4.53,3.87,4.73,4.77,4.62
LLMGraph_Hybrid_KG,4.27,4.33,4.27,4.27,4.40
LLMGraph_Hybrid_KG_Reranker,4.47,4.03,4.67,4.37,4.38
LlamaIndex_Community_Only,3.80,3.33,4.00,3.97,4.08
LlamaIndex_Dense_KG,4.47,3.37,4.53,4.27,4.20
LlamaIndex_Hybrid_KG_Rerank,4.77,4.03,4.90,4.60,4.65
RAG_Advanced_Dense,2.17,1.17,2.13,2.27,2.20
RAG_Advanced_Hybrid,4.80,4.67,4.40,4.53,4.45


In [67]:
df_newest_combi.columns

Index(['script', 'question_id', 'query_type', 'answer_relevance_score',
       'answer_relevance_1to5', 'completeness_score', 'completeness_1to5',
       'correctness_category', 'correctness_coverage',
       'correctness_coverage_1to5', 'correctness_error_severity',
       'faithfulness_score', 'faithfulness_1to5', 'helpfulness_raw_1to5',
       'helpfulness_raw_score', 'helpfulness_final_score',
       'helpfulness_final_1to5', 'helpfulness_justification', 'metrics_json',
       'correctness_rank'],
      dtype='object')